# AERIS - Diagnostic fuite de donnees TB

`tb_cloud_densenet121` obtient AUC = 1.000 exact et Brier = 2.3e-06 sur les 420
images de test. Avant d'y croire, on verifie que le split ne fuit pas.

Les manifestes sont disjoints au niveau des chemins (verifie : 0 chevauchement,
4200 fichiers distincts). Ce qui reste a verifier, c'est le **contenu** :

- doublons stricts, SHA-256, a cheval sur deux lots ;
- quasi-doublons, hachage perceptuel dhash, distance de Hamming <= 5.

Ce notebook ne calcule aucune metrique de performance. Il ne repond qu'a une
question : existe-t-il une image de test dont une copie est a l'entrainement ?


In [ ]:
from pathlib import Path

import hashlib

import numpy as np
import pandas as pd
from PIL import Image

LOTS = ['tuberculose_entrainement.csv', 'tuberculose_validation.csv', 'tuberculose_test.csv']

RACINE_INPUT = Path('/kaggle/input/datasets')


def trouver_dossier(nom_attendu):
    for base in [RACINE_INPUT, Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]:
        if not base.exists():
            continue
        for candidat in sorted(base.glob(f'**/{nom_attendu}')):
            if candidat.is_dir():
                return candidat
    raise FileNotFoundError(nom_attendu)


DATA = trouver_dossier('TB_Chest_Radiography_Database')
print('pool TB', DATA)

candidats = []
for base in [Path('/kaggle/input/datasets'), Path('/kaggle/input'), Path('.')]:
    if not base.exists():
        continue
    try:
        candidats.extend(base.glob('**/tuberculose*.csv'))
    except OSError:
        pass

manifestes = {}
for nom in LOTS:
    trouves = sorted({c for c in candidats if c.name == nom})
    if not trouves:
        raise FileNotFoundError(nom)
    if len(trouves) > 1:
        empreintes = {hashlib.sha256(c.read_bytes()).hexdigest() for c in trouves}
        if len(empreintes) > 1:
            raise RuntimeError(f'manifestes {nom} divergents : ' + ', '.join(str(c) for c in trouves))
        print('copies identiques de', nom, len(trouves), '->', trouves[0])
    manifestes[nom] = pd.read_csv(trouves[0])

index_images = {}
for classe in ('Normal', 'Tuberculosis'):
    dossier = DATA / classe
    for motif in ('*.png', '*.jpeg', '*.jpg', '*.PNG', '*.JPEG'):
        for chemin in dossier.glob(motif):
            index_images[f'{classe}/{chemin.name}'] = chemin
print('images indexees', len(index_images))

lignes = []
for nom, df in manifestes.items():
    for relatif in df['chemin_relatif']:
        cle = '/'.join(str(relatif).split('/')[-2:])
        if cle not in index_images:
            raise FileNotFoundError(f'image absente du pool : {relatif}')
        lignes.append({'lot': nom, 'chemin_relatif': cle, 'etiquette': int(df.loc[df['chemin_relatif'] == relatif, 'etiquette'].iloc[0]), 'chemin': str(index_images[cle])})

table = pd.DataFrame(lignes)
print('references totales', len(table))
print(table.groupby('lot')['etiquette'].agg(['count', 'sum']))


def dhash(chemin, taille=8):
    with Image.open(chemin) as image:
        gris = np.asarray(image.convert('L').resize((taille + 1, taille), Image.BILINEAR), dtype=np.int16)
    bits = (gris[:, 1:] > gris[:, :-1]).flatten()
    valeur = 0
    for b in bits:
        valeur = (valeur << 1) | int(b)
    return valeur


table['sha256'] = [hashlib.sha256(Path(c).read_bytes()).hexdigest() for c in table['chemin']]
table['dhash'] = [dhash(c) for c in table['chemin']]
print('empreintes calculees')
table.to_csv('/kaggle/working/signature_images_tb.csv', index=False)

print()
print('================ DOUBLONS EXACTS (sha256) ================')
par_sha = table.groupby('sha256')['chemin_relatif'].apply(list)
doubles = par_sha[par_sha.apply(len) > 1]
print('groupes d images strictement identiques :', len(doubles))
if len(doubles):
    concerned = set()
    for groupe in doubles:
        for c in groupe:
            concerned.add(c)
    sous = table[table['chemin_relatif'].isin(concerned)]
    print(sous.groupby('lot').size())
    croisements = 0
    for groupe in doubles:
        lots = set(sous.loc[sous['chemin_relatif'].isin(groupe), 'lot'])
        if len(lots) > 1:
            croisements += 1
    print('groupes a cheval sur deux lots (fuite) :', croisements)
    print(sous[['lot', 'chemin_relatif', 'etiquette', 'sha256']].sort_values('sha256').to_string())

print()
print('================ QUASI-DUPLIQUATS (dhash, distance <= 5) ================')
valeurs = [int(v) for v in table['dhash']]
nombre = len(valeurs)
paires_proches = []
for i in range(nombre - 1):
    vi = valeurs[i]
    for j in range(i + 1, nombre):
        if (vi ^ valeurs[j]).bit_count() <= 5:
            paires_proches.append((i, j))

print('paires a distance de hachage <= 5 :', len(paires_proches))
if paires_proches:
    table['index'] = range(nombre)
    concerns = sorted({i for paire in paires_proches for i in paire})
    sous = table.loc[concerns, ['lot', 'chemin_relatif', 'etiquette']]
    print(sous.groupby('lot').size().to_string())
    fuite = sum(1 for i, j in paires_proches if table.loc[i, 'lot'] != table.loc[j, 'lot'])
    print('paires a cheval sur deux lots (fuite) :', fuite)
    for i, j in paires_proches[:60]:
        print('   ', table.loc[i, 'lot'], table.loc[i, 'chemin_relatif'],
              '<->', table.loc[j, 'lot'], table.loc[j, 'chemin_relatif'],
              'distance', (valeurs[i] ^ valeurs[j]).bit_count())
    if len(paires_proches) > 60:
        print('    ...', len(paires_proches) - 60, 'autres paires')
print()
print('================ SEPARABILITE THEORIQUE ================')
etiq = table['etiquette'].to_numpy()
dhash_etiq = {}
for lab in (0, 1):
    sous = table[etiq == lab]['dhash'].to_numpy(dtype=np.uint64)
    dhash_etiq[lab] = sous
print('dhash distincts  Normal  :', len(set(dhash_etiq[0].tolist())), '/', len(dhash_etiq[0]))
print('dhash distincts  Tuberculosis :', len(set(dhash_etiq[1].tolist())), '/', len(dhash_etiq[1]))
print('-> une separation parfaite du dhash entre classes indiquerait que')
print('   les empreintes visuelles suffisent, donc un CNN doit aussi y arriver.')
